# Revisión de experimentos reales

Métrica primaria: F1 macro. Secundarias: accuracy, kappa y matriz de confusión. Compara cinco folds por sujeto del mismo preprocesamiento; no consulta métricas de test.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sleep_staging.config import project_root, local_config, experiment_config, data_config

root = project_root()
cfg = experiment_config(root / "configs/experiments/rf-eeg.yaml")
data = data_config(root / cfg.data_config)
settings = local_config()
assert data.source in {"sleep-cassette", "sleep-telemetry"}
print("Dataset real:", data.name, "| Tarea:", cfg.task)


Dataset real: sc-pilot | Tarea: task_5


In [2]:
from mlflow.tracking import MlflowClient
from sleep_staging.tracking import require_tracking
require_tracking(settings.tracking_uri)
client = MlflowClient(tracking_uri=settings.tracking_uri)
experiment = client.get_experiment_by_name(cfg.name)
if experiment is None:
    raise RuntimeError("Ejecuta primero los experimentos RF y CNN del piloto.")
runs = client.search_runs([experiment.experiment_id], filter_string="tags.dataset = 'sc-pilot' AND attributes.status = 'FINISHED'", order_by=["attributes.start_time DESC"])
summaries = []
for run in runs:
    path = root / "var/reports" / run.info.run_id / "summary.json"
    if path.exists():
        summaries.append(json.loads(path.read_text()))
if not summaries:
    raise RuntimeError("No hay informes de corridas reales completadas.")
version = summaries[0]["preprocessing_id"]
splits_version = summaries[0]["splits_path"]
compatible = [
    s
    for s in summaries
    if s["preprocessing_id"] == version
    and s["task"] == cfg.task
    and s.get("config", {}).get("channels") == cfg.channels
    and s["splits_path"] == splits_version
]
# Nota (fix): ademas de preprocessing_id/task/channels, se exige splits_path
# identico. Sin este filtro, dos corridas con la misma configuracion pero
# particiones de sujetos distintas (p. ej. piloto de 8 sujetos vs. campana de
# 62) podian mezclarse en el mismo leaderboard, invalidando la comparacion
# entre modelos (hallazgo de la autorrevision del PR #1).
rows = [{**row, "run_id": s["run_id"]} for s in compatible for row in s["leaderboard"]]
leaderboard = pd.DataFrame(rows).drop_duplicates("model", keep="first").sort_values("f1_macro", ascending=False)
display(leaderboard[["model", "f1_macro", "f1_std", "accuracy", "elapsed_seconds", "run_id"]])
winner = leaderboard.iloc[0]
print(f"Winner model: {winner.model}; F1 macro CV = {winner.f1_macro:.4f} +/- {winner.f1_std:.4f}")


/Users/adrianalarcon/Documents/uniandes/maia_despliegue_tesis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,model,f1_macro,f1_std,accuracy,elapsed_seconds,run_id
2,rf,0.583047,0.139766,0.780000,0.625391,3c246ec4ec13420a9388a4511e5fddb6
0,cnn,0.283689,0.045452,0.462500,3.037737,0e56c65098694e01b43109aa0737a2fe
1,dummy,0.165094,0.004102,0.703333,0.208768,0e56c65098694e01b43109aa0737a2fe


Winner model: rf; F1 macro CV = 0.5830 +/- 0.1398


## Modelo ganador y recarga

Se selecciona por F1 macro de desarrollo. El ejemplo usa épocas de desarrollo para comprobar la recarga.

In [3]:
import mlflow
from sleep_staging.experiments import waveform_frame
from sleep_staging.inference import predict_features
selected = next(s for s in compatible if s["run_id"] == winner.run_id)
metadata = mlflow.models.get_model_info(selected["model_uri"]).metadata
display(pd.Series(metadata))
oof = pd.read_parquet(root / "var/reports" / selected["run_id"] / "oof.parquet")
ids = oof[oof.model.eq(winner.model)].epoch_id.head(5).tolist()
if metadata["input_kind"] == "waveform":
    prepared = root / "data/interim" / selected["preprocessing_id"]
    epochs = pd.read_parquet(prepared / "epochs.parquet")
    subset = epochs[epochs.epoch_id.isin(ids)]
    inputs = waveform_frame(subset, prepared, cfg.channels[0])
    inputs.insert(0, "epoch_id", subset.epoch_id.to_numpy())
else:
    features = pd.read_parquet(selected["features_path"])
    inputs = features[features.epoch_id.isin(ids)].copy()
display(predict_features(selected["model_uri"], inputs, settings.tracking_uri))
example_path = root / "var/reports" / selected["run_id"] / "example-input.parquet"
inputs.to_parquet(example_path, index=False)
print("Model URI:", selected["model_uri"])
print("Input de ejemplo:", example_path)


channels                                                 [eeg_fpz_cz]
feature_columns     [eeg_fpz_cz__eeg__mean, eeg_fpz_cz__eeg__std, ...
input_kind                                                   features
preprocessing       {'eeg_high': 35, 'eeg_low': 0.3, 'max_epochs_p...
preprocessing_id                                 dded564a47dfb085625a
task                                                           task_5
dtype: object

,epoch_id,prediction,prob_N1,prob_N2,prob_N3,prob_REM,prob_W
1440,SC4061:0,W,0.062766,0.097749,0.000000,0.109097,0.730387
1441,SC4061:33000,W,0.000000,0.000000,0.000000,0.000000,1.000000
1442,SC4061:69000,W,0.012500,0.014323,0.000000,0.000000,0.973177
1443,SC4061:102000,W,0.000000,0.000000,0.013393,0.000000,0.986607
1444,SC4061:138000,W,0.000000,0.000000,0.000000,0.013021,0.986979


Model URI: models:/m-7ce6f9a435b240968e4042b7ff7116be
Input de ejemplo: /Users/adrianalarcon/Documents/uniandes/maia_despliegue_tesis/var/reports/3c246ec4ec13420a9388a4511e5fddb6/example-input.parquet


## Siguiente iteración

El ganador corresponde al piloto de ocho sujetos y sus cinco folds. Completar Sleep Cassette, congelar el protocolo y evaluar el test reservado en una etapa posterior. Revisar N1, cobertura y cambios de cohorte antes de ampliar la complejidad.